# dataloader-pin-memory-workers — worked example 2: Drive a DataLoader with a SubsetRandomSampler

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `dataloader-pin-memory-workers`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

`shuffle=True` and a custom `sampler` are mutually exclusive — passing both raises. To iterate only part of a dataset in random order you pass `sampler=SubsetRandomSampler(indices)` and leave `shuffle` unset. The sampler decides *which* indices are drawn and in what order; the loader still groups them into `batch_size` chunks.

## Worked solution

Goal: build a loader that visits only the first 12 of 20 items, in a shuffled order, using a sampler.

1. **Make the dataset.** `TensorDataset(X)` over `N=20` rows of shape `(3,)`.
2. **Pick the subset indices.** We choose indices `0..11` — a `SubsetRandomSampler` will permute these 12 each epoch.
3. **Why no `shuffle=True`?** A sampler already controls order; combining it with `shuffle=True` is illegal and raises `ValueError`. So we omit `shuffle` (defaults to `False`) and pass `sampler=`.
4. **Seed for determinism.** `SubsetRandomSampler` uses the global torch RNG, so we `t.manual_seed(0)` before iterating to make the permutation reproducible.
5. **Verify coverage.** Iterating one epoch should touch exactly the 12 chosen indices, each once. We recover which rows appeared by matching values back to `X` and confirm the set equals `{0..11}` — order will differ from sorted, proving the shuffle.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset, SubsetRandomSampler

def make_subset_loader(dataset, indices, batch_size):
    sampler = SubsetRandomSampler(indices)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        sampler=sampler,
        num_workers=0,
        pin_memory=False,
    )

t.manual_seed(0)
X = (t.arange(20).reshape(20, 1) * t.ones(1, 3)).float()  # row i is all-i, shape (20,3)
ds = TensorDataset(X)
idx = list(range(12))
loader = make_subset_loader(ds, idx, batch_size=4)

seen = []
for (batch,) in loader:
    seen.extend(int(row[0].item()) for row in batch)
print('count:', len(seen), '(expected 12)')
print('unique sorted:', sorted(set(seen)))
print('order shuffled (not 0..11):', seen != sorted(seen))